# Handling missing values

[Thomas Moreau](https://tommoral.github.io/about.html), adapted from the labs of
[Marine Le Morvan](https://marinelm.github.io/)

---

Real datasets have holes: a sensor failed, a patient skipped a test, a client did not
answer a question. Most estimators in `scikit-learn` refuse `NaN` as input, so you
*have* to make a choice. This notebook is about making that choice on purpose.

We first look at the problem: what missing values look like in real data, and why
*the reason* a value is missing matters. Then we review the main families of
solutions, and finally we compare them on a prediction task.

## Table of contents

* [1 The problem](#problem)
    * [1.1 Missing values in the wild](#problem_wild)
    * [1.2 The mask and the three missingness mechanisms](#problem_mechanisms)
* [2 The families of solutions](#families)
    * [2.1 Dropping](#families_drop)
    * [2.2 Imputing](#families_impute)
    * [2.3 Keeping the mask](#families_mask)
    * [2.4 Models that handle missing values natively](#families_native)
* [3 What works for prediction?](#prediction)
    * [3.1 A linear problem with MCAR values](#prediction_mcar)
    * [3.2 When missingness is informative](#prediction_mnar)
    * [3.3 A non-linear problem: imputing well is not enough](#prediction_nonlinear)
    * [3.4 Back to the Titanic](#prediction_titanic)
* [4 Take-home messages](#takehome)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, IterativeImputer, KNNImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score

RANDOM_STATE = 42
plt.rcParams["figure.figsize"] = (5, 4)

# 1 The problem <a class="anchor" id="problem"></a> [↑](#Table-of-contents)

## 1.1 Missing values in the wild <a class="anchor" id="problem_wild"></a> [↑](#Table-of-contents)

The Titanic passenger list is a small dataset with plenty of holes. The task is to
predict whether a passenger survived.

In [ ]:
titanic = pd.read_csv("data/titanic.csv")
titanic.head()

The first thing to do with a new dataset is to measure how much is missing, per column.

In [ ]:
titanic.isna().mean().sort_values(ascending=False).plot.barh()
plt.xlabel("Fraction of missing values")
plt.show()

Looking at the *pattern* of missing values (one row per passenger, black = missing)
often tells more than the rates. The file is sorted by class, and the cabin is
missing in blocks: it was mostly recorded for first class passengers.

In [ ]:
plt.figure(figsize=(6, 4))
plt.imshow(titanic.isna(), aspect="auto", cmap="gray_r", interpolation="nearest")
plt.xticks(range(titanic.shape[1]), titanic.columns, rotation=90)
plt.ylabel("Passenger")
plt.show()

**Missingness is information.** Is the survival rate the same whether the cabin (or
the age) is known or not?

In [ ]:
pd.DataFrame({
    col: titanic.groupby(titanic[col].isna())["survived"].mean()
    for col in ["cabin", "age"]
}).rename(index={False: "observed", True: "missing"})

Passengers with no recorded cabin survived half as often. The cabin was mostly recorded
for first class passengers, and the missing ages are mostly in third class. Whatever
we do with these `NaN`, we should not throw this signal away.

Two more warnings before modelling:

* **Not all missing values are `NaN`.** Check for sentinel values such as `-999`, `""`,
  `"?"` or `0`. Here, 17 passengers have `fare == 0`: free tickets (several worked for
  the shipbuilder) or an unknown fare? Only domain knowledge can tell.
* **Missingness can leak the target.** `body` is the identification number of a
  recovered body: it is observed *only* for passengers who died. `boat` is the
  lifeboat. Both are recorded after the event and must be dropped.

In [ ]:
print(f"fare == 0: {(titanic['fare'] == 0).sum()} passengers")
pd.crosstab(titanic["body"].isna(), titanic["survived"])

## 1.2 The mask and the three missingness mechanisms <a class="anchor" id="problem_mechanisms"></a> [↑](#Table-of-contents)

For a data matrix $X$, the **missingness indicator** (or **mask**) $M$ has the same
shape, with $M_{ij} = 1$ when $X_{ij}$ is missing:

$$
X = \begin{pmatrix} 1.2 & \texttt{NA} & 0.2 \\ -0.3 & \texttt{NA} & \texttt{NA} \\ 0.5 & 0.4 & \texttt{NA}\end{pmatrix}
\qquad
M = \begin{pmatrix} 0 & 1 & 0 \\ 0 & 1 & 1 \\ 0 & 0 & 1\end{pmatrix}
$$

[Rubin (1976)](https://doi.org/10.1093/biomet/63.3.581) classified missing values by
how $M$ depends on $X$:

* **Missing Completely At Random (MCAR)**: $P(M | X) = P(M)$. The mask is pure noise.
  *A sensor fails at random times.*
* **Missing At Random (MAR)**: $P(M | X) = P(M | X_{obs})$. The mask depends only on
  values that *are* observed. *A blood test is skipped for young patients, and the age
  is always recorded.*
* **Missing Not At Random (MNAR)**: everything else, typically when the mask depends
  on the missing value itself. *High earners do not report their income.*

To see the difference, we use a toy dataset: two correlated Gaussian features, and a
target that is a linear function of the first one. Only `x1` has missing values.

In [ ]:
def generate_data(n_samples, rng=RANDOM_STATE):
    rng = np.random.default_rng(rng)
    X = rng.multivariate_normal([0, 0], [[1, 0.9], [0.9, 1]], size=n_samples)
    y = X[:, 0] + 0.1 * rng.standard_normal(n_samples)
    return X, y


def add_missing_values(X, mechanism, rng=RANDOM_STATE + 1):
    """Hide about half of the values of x1, with the given mechanism."""
    rng = np.random.default_rng(rng)
    if mechanism == "MCAR":
        proba = np.full(len(X), 0.5)
    elif mechanism == "MAR":
        proba = 1 / (1 + np.exp(-3 * X[:, 1]))  # depends on x2, always observed
    elif mechanism == "MNAR":
        proba = 1 / (1 + np.exp(-3 * X[:, 0]))  # depends on x1 itself
    X = X.copy()
    X[rng.random(len(X)) < proba, 0] = np.nan
    return X

In [ ]:
X_full, y_full = generate_data(1000)

fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=True)
for ax_scatter, ax_hist, mechanism in zip(*axes, ["MCAR", "MAR", "MNAR"]):
    X = add_missing_values(X_full, mechanism)
    missing = np.isnan(X[:, 0])
    ax_scatter.scatter(*X_full[missing].T, color=".8", s=10, label="x1 missing")
    ax_scatter.scatter(*X_full[~missing].T, s=10, label="x1 observed")
    ax_scatter.set(title=mechanism, ylabel="x2")
    ax_hist.hist(X_full[:, 0], bins=30, color=".8", label="all values")
    ax_hist.hist(X[~missing, 0], bins=30, alpha=0.7, label="observed values")
    ax_hist.set(xlabel="x1")
axes[0, 0].legend()
axes[1, 0].legend()
plt.tight_layout()

Under MCAR, the observed values of `x1` are a random subsample: same distribution as
the full data. Under MAR and MNAR, the large values of `x1` are missing more often, so
**the observed values are a biased sample**.

The difference between MAR and MNAR is subtle on these plots. Under MAR, the bias can
be corrected using `x2`: among passengers with the same `x2`, the observed and missing
values of `x1` have the same distribution. Under MNAR, nothing in the observed data
tells us how the missing values differ from the observed ones.

This is a well-known effect in surveys. In the 2012 US census, the median household
income was about 51,000\$. In a survey where only 66% of the people answered, the
median of the answers dropped to 39,000\$.

Unfortunately, **the mechanism cannot be tested from the data**: MNAR data looks exactly
like some MAR data. It is an assumption that you justify with domain knowledge.

# 2 The families of solutions <a class="anchor" id="families"></a> [↑](#Table-of-contents)

We now review four ways to deal with the missing values in `x1`, using the MAR version
of the toy data.

In [ ]:
X_mar = add_missing_values(X_full, "MAR")
missing = np.isnan(X_mar[:, 0])


def plot_imputation(X_imputed, title):
    plt.figure()
    plt.scatter(*X_full.T, color=".8", s=10, label="original data")
    plt.scatter(*X_imputed[missing].T, marker="x", s=10, label="imputed values")
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.legend()

## 2.1 Dropping <a class="anchor" id="families_drop"></a> [↑](#Table-of-contents)

The simplest option is *complete-case analysis*: drop every row with a missing value
(`df.dropna()`), or every column with too many of them.

* It throws away data: on the Titanic, dropping every incomplete row keeps none of the
  1309 passengers, because `body` is almost always missing.
* Unless the data is MCAR, the remaining rows are a biased sample (see the histograms
  above).
* **It does not work at prediction time**: a model must give a prediction for a new
  passenger even if their age is unknown.

In [ ]:
print(f"Titanic rows without any missing value: {len(titanic.dropna())}")
print(f"Toy data: mean of x1 is {X_full[:, 0].mean():.2f}, "
      f"mean of the observed x1 is {np.nanmean(X_mar[:, 0]):.2f}")

## 2.2 Imputing <a class="anchor" id="families_impute"></a> [↑](#Table-of-contents)

Imputing means replacing each missing value by a plausible one, to get a complete
matrix that any model accepts. `scikit-learn` offers:

* `SimpleImputer`: a constant per column (`mean`, `median`, `most_frequent` or a fixed
  value). Fast, but it ignores the other features.
* `KNNImputer`: the mean of the nearest neighbours, found using the observed features.
* `IterativeImputer`: each feature with missing values is regressed on the others, in
  turn, until convergence (MICE-like). It predicts the missing values from the observed
  ones. It is the strongest of the three, but its cost grows quickly with the number of
  features.

In [ ]:
plot_imputation(SimpleImputer(strategy="mean").fit_transform(X_mar), "Mean imputation")
plot_imputation(IterativeImputer().fit_transform(X_mar), "Iterative imputation")

Mean imputation puts every missing value on one vertical line and breaks the link
between `x1` and `x2`. The iterative imputer predicts `x1` from `x2` and places the
imputed values along the main axis of the data, which is the best guess
($\mathbb{E}[x_1 | x_2]$), but without the spread of the true values.

**An imputer is fitted on the training data only**, like any other preprocessing step.
Imputing the whole dataset before a cross-validation leaks information from the test
folds. Put the imputer in a `Pipeline`, and `cross_val_score` takes care of it.

How bad is the leak? We compare the two on small and wide data (100 samples, 30
features, half of the values missing), where the imputer has much to learn.

In [ ]:
from sklearn.model_selection import KFold

rng = np.random.default_rng(0)
X_wide = rng.standard_normal((100, 30))
y_wide = X_wide @ rng.standard_normal(30) + rng.standard_normal(100)
X_wide[rng.random(X_wide.shape) < 0.5] = np.nan
cv_wide = KFold(5, shuffle=True, random_state=RANDOM_STATE)

# Right: the imputer is fitted on the training folds only.
in_pipeline = make_pipeline(IterativeImputer(), RidgeCV())
print(f"Imputer in the pipeline: "
      f"R2 = {cross_val_score(in_pipeline, X_wide, y_wide, cv=cv_wide).mean():.2f}")

# Wrong: the imputer sees the test folds.
X_leak = IterativeImputer().fit_transform(X_wide)
print(f"Imputation before the CV: "
      f"R2 = {cross_val_score(RidgeCV(), X_leak, y_wide, cv=cv_wide).mean():.2f}")

# Worse: the target is still in the table when imputing.
X_leak_y = IterativeImputer().fit_transform(np.c_[X_wide, y_wide])[:, :-1]
print(f"Imputation with y in the table: "
      f"R2 = {cross_val_score(RidgeCV(), X_leak_y, y_wide, cv=cv_wide).mean():.2f}")

Imputing before the cross-validation makes the model look better than it is. When
the target is in the table, the imputed values are predicted from $, and the score
is almost perfect. This happens easily with `IterativeImputer().fit_transform(df)`
on a dataframe that still holds the target. With mean imputation, the leak is a
single number per column and is usually too small to see, but the rule is the same.

<div class="alert alert-success">
    <p><b>Exercise</b>:</p>

Assume we want to estimate the mean of `x1`, which is 0 in the full data. For each
mechanism (`MCAR`, `MAR`, `MNAR`), generate 10,000 samples with missing values
and compare three estimates:

1. the mean of the observed values (complete-case),
2. the mean of `x1` after mean imputation,
3. the mean of `x1` after iterative imputation.

Which estimates are biased, and under which mechanism? How does this match the
discussion of Section 1.2?

</div>

In [ ]:
# Your work.

In [ ]:
# %load solutions/solution_1.py

## 2.3 Keeping the mask <a class="anchor" id="families_mask"></a> [↑](#Table-of-contents)

Imputation hides the fact that a value was missing, and we saw on the Titanic that this
fact can be informative. All `scikit-learn` imputers have an `add_indicator` option
that appends the mask $M$ as extra binary columns, for the features that had missing
values during `fit`.

In [ ]:
SimpleImputer(add_indicator=True).fit_transform(X_mar)[:5]

The last column is $M_1$, the missingness indicator of `x1`. The model can now learn a
different behaviour for imputed and observed values.

## 2.4 Models that handle missing values natively <a class="anchor" id="families_native"></a> [↑](#Table-of-contents)

Some models accept `NaN` directly. Trees do it naturally: at each split on a feature
with missing values, the samples with `NaN` are sent to the left or the right child,
whichever reduces the loss the most. This is called *Missing Incorporated in Attribute*
(MIA). It learns at the same time how to impute and how to use the mask, with no
extra step.

In `scikit-learn`, `HistGradientBoosting{Regressor,Classifier}`, `RandomForest*` and
`DecisionTree*` support it, as do XGBoost and LightGBM. The tabular foundation models
of Day 3 (TabPFN, TabICL) also accept `NaN`.

In [ ]:
HistGradientBoostingRegressor().fit(X_mar, y_full).score(X_mar, y_full)

| Strategy | Example | Pros | Cons |
|---|---|---|---|
| Drop | `df.dropna()` | trivial | loses data, biased, impossible at test time |
| Impute | `SimpleImputer`, `IterativeImputer` | any model downstream | hides that the value was missing |
| Impute + mask | `add_indicator=True` | keeps the missingness signal | more features |
| Native | `HistGradientBoosting*` | no preprocessing, often the best | only some models |

# 3 What works for prediction? <a class="anchor" id="prediction"></a> [↑](#Table-of-contents)

For inference (estimating a mean, a coefficient), the goal is to recover the full data
distribution, and the mechanism is central. For prediction, the goal is different: we
want the best possible prediction of $y$ *given what we observe*, the mask included.
A good imputation is a means, not the goal.

We compare a few pipelines with cross-validation, on larger data.

In [ ]:
def compare(models, X, y, title):
    scores = pd.DataFrame({
        name: cross_val_score(model, X, y, cv=5) for name, model in models.items()
    })
    plt.figure(figsize=(6, 3))
    sns.boxplot(data=scores, orient="h")
    plt.xlabel("$R^2$")
    plt.title(title)
    plt.show()
    return scores.mean().round(3)


X_large, y_large = generate_data(20_000)

## 3.1 A linear problem with MCAR values <a class="anchor" id="prediction_mcar"></a> [↑](#Table-of-contents)

In [ ]:
models = {
    "Mean imputation + Ridge": make_pipeline(SimpleImputer(), RidgeCV()),
    "Iterative imputation + Ridge": make_pipeline(IterativeImputer(), RidgeCV()),
    "Mean imputation + HGB": make_pipeline(
        SimpleImputer(), HistGradientBoostingRegressor()
    ),
    "HGB (native)": HistGradientBoostingRegressor(),
}
compare(models, add_missing_values(X_large, "MCAR"), y_large, "Linear target, MCAR")

With a linear model, the quality of the imputation matters: mean imputation breaks the
linear relation between the features and the target, while the iterative imputer keeps
it. A flexible model such as gradient boosting compensates for the poor imputation: it
learns that `x1 = mean` is a special value and uses `x2` instead.

## 3.2 When missingness is informative <a class="anchor" id="prediction_mnar"></a> [↑](#Table-of-contents)

<div class="alert alert-success">
    <p><b>Exercise</b>:</p>

Run the same comparison on MNAR data, where the large values of `x1` are the ones that
go missing.

1. How do the four pipelines compare now? Why does the iterative imputer lose its
   advantage?
2. Add the mask to the two Ridge pipelines (`add_indicator=True`). Which pipeline
   benefits most from it, and why?

</div>

In [ ]:
# Your work.

In [ ]:
# %load solutions/solution_2.py

## 3.3 A non-linear problem: imputing well is not enough <a class="anchor" id="prediction_nonlinear"></a> [↑](#Table-of-contents)

A tempting strategy: train a model on clean, complete data, then impute the new data
well and feed it to that model. We test it on a target with curvature, $y^2$, with MCAR
values in both features.

In [ ]:
X_train_full, y_train_full = generate_data(20_000, rng=0)
X_test_full, y_test_full = generate_data(20_000, rng=1)
y_train_full, y_test_full = y_train_full**2, y_test_full**2
X_train = add_missing_values(X_train_full, "MCAR", rng=2)
X_test = add_missing_values(X_test_full, "MCAR", rng=3)
rng = np.random.default_rng(4)
for X in (X_train, X_test):
    X[rng.random(len(X)) < 0.5, 1] = np.nan  # x2 is now missing too

full_data_model = HistGradientBoostingRegressor().fit(X_train_full, y_train_full)
print(f"Full data model on complete test data: "
      f"R2 = {full_data_model.score(X_test_full, y_test_full):.3f}")

imputer = IterativeImputer().fit(X_train)
print(f"Full data model on imputed test data: "
      f"R2 = {full_data_model.score(imputer.transform(X_test), y_test_full):.3f}")

na_model = HistGradientBoostingRegressor().fit(X_train, y_train_full)
print(f"Model trained with missing values: "
      f"R2 = {na_model.score(X_test, y_test_full):.3f}")

The model trained on complete data is near perfect when there are no missing values,
but it is clearly beaten on imputed data by a model trained with the missing values.

The imputer predicts $\mathbb{E}[x_1 | x_2]$, so the full data model predicts
$f(\mathbb{E}[x_1 | x_2])$. The best prediction is $\mathbb{E}[f(x_1) | x_2]$, and the
two differ as soon as $f$ is not linear: the uncertainty about $x_1$ turns into a bias.
**Train the model on data with the same missing values as at prediction time.**

## 3.4 Back to the Titanic <a class="anchor" id="prediction_titanic"></a> [↑](#Table-of-contents)

We drop the columns that leak the target or are free text, and keep the deck (first
letter of the cabin) as a categorical feature.

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold

X_titanic = titanic.drop(columns=["survived", "name", "ticket", "boat", "body",
                                  "home.dest"])
X_titanic["cabin"] = X_titanic["cabin"].str[0]
y_titanic = titanic["survived"]

numerical = ["pclass", "age", "sibsp", "parch", "fare"]
categorical = ["sex", "embarked", "cabin"]

# A missing category is just one more category.
categorical_pipe = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="missing"),
    OneHotEncoder(handle_unknown="ignore"),
)


def titanic_model(numerical_pipe, classifier):
    preprocessor = make_column_transformer(
        (numerical_pipe, numerical), (categorical_pipe, categorical)
    )
    return make_pipeline(preprocessor, classifier)


model = titanic_model(
    make_pipeline(SimpleImputer(), StandardScaler()), LogisticRegression()
)
# The file is sorted by class: shuffle the folds.
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
cross_val_score(model, X_titanic, y_titanic, cv=cv, scoring="roc_auc").mean()

<div class="alert alert-success">
    <p><b>Exercise</b>:</p>

Using `titanic_model` and the same `cv`, compare (ROC AUC):

1. mean imputation + logistic regression (above),
2. the same with the missingness indicator,
3. iterative imputation + logistic regression,
4. `HistGradientBoostingClassifier` with no numerical imputation (use
   `"passthrough"` as numerical pipeline).

How large are the differences? We saw in Section 1.1 that a missing age is
associated with a lower survival rate: why does the mask not help here?

</div>

In [ ]:
# Your work.

In [ ]:
# %load solutions/solution_3.py

# 4 Take-home messages <a class="anchor" id="takehome"></a> [↑](#Table-of-contents)

* Look at **how much** is missing and **how**: rates, patterns, sentinel values, and
  the link between missingness and the target.
* The mechanism (MCAR, MAR, MNAR) cannot be tested from the data. It decides whether
  the observed data is a biased sample.
* For prediction, **fit the imputer on the training data only**, in a pipeline, and
  train the model on data with missing values like the ones it will see.
* **Better imputation helps, but often only marginally.** A flexible model
  compensates for a simple imputation.
* **Add the mask** (`add_indicator=True`): it is cheap and helps as soon as
  missingness is informative, and benchmarks show it helps even under MCAR.
* **Gradient boosted trees with native `NaN` support** are a very strong default.

### To go further

* J. Josse et al. (2019), [On the consistency of supervised learning with missing values](https://arxiv.org/abs/1902.06931).
* M. Le Morvan et al. (2021), [What's a good imputation to predict with missing values?](https://arxiv.org/abs/2106.00311)
* A. Perez-Lebel et al. (2022), [Benchmarking missing-values approaches for predictive models on health databases](https://doi.org/10.1093/gigascience/giac013).
* The `scikit-learn` user guide on [imputation](https://scikit-learn.org/stable/modules/impute.html).
* For inference with missing values (inverse probability weighting, multiple
  imputation), the [R-miss-tastic](https://rmisstastic.netlify.app/) resource page.